# Sprint 3: Segmentation (Phase A)
## K-Means Clustering Optimization & Visualization

This notebook evaluates the optimal number of clusters ($K$) for customer segmentation using the **Elbow Method** and **Silhouette Scores**. It also generates PCA scatter plots to visualize the cluster boundaries.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Set project root in path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import config
from src.clustering.rfm_scoring import apply_rfm_scoring
from src.clustering.kmeans_baseline import preprocess_features

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

### 1. Load Data & Apply Preprocessing

In [ ]:
data_path = os.path.join(config.PROCESSED_DATA_DIR, "customer_feature_matrix.csv")

if not os.path.exists(data_path):
    print(f"Could not find feature matrix at {data_path}. Generating dummy data for notebook demonstration...")
    # Generate dummy data for the purpose of the notebook if not run yet
    df = pd.DataFrame({
        'customer_id': range(1000),
        'recency_days': np.random.randint(1, 365, 1000),
        'frequency_total': np.random.randint(1, 50, 1000),
        'monetary_value_total': np.random.lognormal(mean=5, sigma=1, size=1000),
        'avg_order_value': np.random.lognormal(mean=3, sigma=0.5, size=1000),
        'spend_last_30d': np.random.randint(0, 500, 1000)
    })
else:
    df = pd.read_csv(data_path)

print(f"Loaded dataset with shape: {df.shape}")

# Preprocess features (imputation and scaling)
X_scaled, feature_names = preprocess_features(df, is_training=True)
print(f"Scaled features shape: {X_scaled.shape}")

### 2. Determine Optimal $K$ (Elbow & Silhouette)

In [ ]:
k_range = range(2, 11)
inertias = []
silhouette_scores = []

print("Evaluating K values from 2 to 10...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=config.RANDOM_SEED, n_init='auto')
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    print(f"K={k} -> Inertia: {kmeans.inertia_:.2f} | Silhouette: {silhouette_scores[-1]:.4f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Elbow Curve (Inertia)
color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (Within-cluster Sum of Squares)', color=color)
ax1.plot(k_range, inertias, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

# Silhouette Score
ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, silhouette_scores, marker='s', color=color, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means Optimization: Elbow Curve vs Silhouette Score', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_dir = os.path.join(config.BASE_DIR, 'plots', 'clustering')
os.makedirs(plot_dir, exist_ok=True)
plt.savefig(os.path.join(plot_dir, 'optimal_k_evaluation.png'), dpi=300)
plt.show()

### 3. Visualizing the Clusters with PCA

In [ ]:
# Let's assume K=5 based on standard RFM segments
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=config.RANDOM_SEED, n_init='auto')
df['cluster_label'] = kmeans.fit_predict(X_scaled)

# Reduce dimensions to 2D for visualization
pca = PCA(n_components=2, random_state=config.RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

df['pca_1'] = X_pca[:, 0]
df['pca_2'] = X_pca[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='pca_1', y='pca_2',
    hue='cluster_label',
    palette='viridis',
    data=df,
    alpha=0.7,
    s=60
)

plt.title(f'Customer Segments (K={optimal_k}) Projected on 2D PCA Space', fontsize=14, fontweight='bold')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'pca_cluster_scatter.png'), dpi=300)
plt.show()